# Import important python packages 

In [ ]:
import xarray as xr 
import numpy as np 
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.ticker import MultipleLocator
from matplotlib.colors import BoundaryNorm
import os
import shutil
import geopandas as gpd

## Data path 

In [ ]:
datapath = "/data/projects/WWRF-NRT/30YEAR-REFORECAST/MODE_verification/sshamsian/Forecast_verification_sshamsian/nws_precip_1day_20240122_conus.nc"
ds = xr.open_dataset(datapath)

### Code for plotting the data over San diego 

In [ ]:
precip_attrs = ds['observation'].attrs
 
# Native polar stereographic projection of the grid (from proj4 attrs)
# +proj=stere +lat_0=90 +lat_ts=60 +lon_0=-105 +a=6371200 +b=6371200
src_crs = ccrs.Stereographic(
    central_latitude=90,
    central_longitude=-105,
    true_scale_latitude=60,
    globe=ccrs.Globe(ellipse=None, semimajor_axis=6371200, semiminor_axis=6371200)
)
 
# Bounding box around San Diego County (lon/lat)
lon_min, lon_max = -117.8, -116.0
lat_min, lat_max = 32.4, 33.6
 
# Use cartopy/pyproj (under the hood) just to find the x/y bounds for subsetting.
# transform_points handles the lon/lat -> native x/y conversion for us.
corners_lonlat = np.array([
    [lon_min, lat_min],
    [lon_max, lat_min],
    [lon_min, lat_max],
    [lon_max, lat_max],
])
corners_xy = src_crs.transform_points(ccrs.PlateCarree(), corners_lonlat[:, 0], corners_lonlat[:, 1])
x0, x1 = corners_xy[:, 0].min(), corners_xy[:, 0].max()
y0, y1 = corners_xy[:, 1].min(), corners_xy[:, 1].max()
buf = 5000  # meters
 
sub = ds.sel(x=slice(x0 - buf, x1 + buf), y=slice(y1 + buf, y0 - buf))
precip = sub['observation']
print("Subset shape:", precip.shape, "max in:", float(precip.max()))
os.environ['SHAPE_RESTORE_SHX'] = 'YES'
shp_dir = '/data/projects/WWRF-NRT/30YEAR-REFORECAST/MODE_verification/sshamsian/Forecast_verification_sshamsian/'
counties = gpd.read_file(f'{shp_dir}/Counties.shp')
counties = counties.set_crs('EPSG:3857')
counties_lonlat = counties.to_crs('EPSG:4326')
 
# --- Plot ---
fig = plt.figure(figsize=(8, 7.5))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
 
levels = [0, 0.01, 0.1, 0.25, 0.5, 0.75, 1, 1.5, 2, 3, 4, 5]
cmap = plt.get_cmap('YlGnBu')
norm = BoundaryNorm(levels, cmap.N)
 
# Pass the native x/y coords + native CRS as `transform`; cartopy reprojects
mesh = ax.pcolormesh(
    sub['x'], sub['y'], precip.values,
    transform=src_crs, cmap=cmap, norm=norm, shading='auto'
)
 
# County outlines, drawn on top of the precip mesh in their native lon/lat CRS
ax.add_geometries(
    counties_lonlat.geometry, crs=ccrs.PlateCarree(),
    facecolor='none', edgecolor='black', linewidth=0.8, zorder=4
)
 
# San Diego marker
sd_lon, sd_lat = -117.1611, 32.7157
ax.plot(sd_lon, sd_lat, marker='*', color='red', markersize=16,
        markeredgecolor='black', transform=ccrs.PlateCarree(), zorder=5)
ax.text(sd_lon + 0.05, sd_lat + 0.03, 'San Diego', transform=ccrs.PlateCarree(),
        fontsize=11, fontweight='bold')
 
# Manual ticks/labels instead of ax.gridlines(draw_labels=True, ...).
ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.5, linestyle='--')
ax.set_xticks(np.arange(np.floor(lon_min), np.ceil(lon_max) + 0.01, 0.25), crs=ccrs.PlateCarree())
ax.set_yticks(np.arange(np.floor(lat_min), np.ceil(lat_max) + 0.01, 0.15), crs=ccrs.PlateCarree())
ax.xaxis.set_major_formatter(LongitudeFormatter())
ax.yaxis.set_major_formatter(LatitudeFormatter())
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
 
cbar = plt.colorbar(mesh, ax=ax, orientation='vertical', pad=0.05, shrink=0.85, ticks=levels)
cbar.set_label('Precipitation (inches)')
 
data_time = ds.attrs.get('data_time', 'N/A')
ax.set_title(f'24-Hour Observed Precipitation — San Diego County\nNWS Data, ending {data_time}', fontsize=12)
 
plt.tight_layout()
print("Done.")